In [1]:
pip install -U langchain langchain-openai

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 15.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.54
    Uninstalling langchain-core-0.3.54:
      Successfully uninstalled langchain-core-0.3.54
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.23
    Uninstalling langchain-0.3.23:
      Successfully uninstalled langchain-0.3.23
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv()

import os


In [4]:

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="o4-mini")
llm.invoke("Hello, world!")

AIMessage(content='Hello there! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 10, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'o4-mini-2025-04-16', 'system_fingerprint': None, 'id': 'chatcmpl-BQz6rsbJfudrOrsxtdjf1F18cD16Q', 'finish_reason': 'stop', 'logprobs': None}, id='run-74effe07-b943-4ba8-a36c-0570e0d4e7ab-0', usage_metadata={'input_tokens': 10, 'output_tokens': 29, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
from langgraph.graph import StateGraph, END
import random
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
from typing import List, Dict, Any
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser

In [7]:
prompt_quiz = ChatPromptTemplate.from_messages([
    ("system",
     """Eres un experto en educación. Evalúa las siguientes respuestas del usuario a preguntas de probabilidad y estadística.
Para cada respuesta, califica de 0 a 5 (donde 0 es incorrecta y 5 es perfecta), explica brevemente la calificación.

Devuelve la respuesta SOLO en formato JSON con la siguiente estructura:
{{
  "resultados": [puntaje1, puntaje2, ...],
  "detalle": [
    {{
      "pregunta": "...",
      "respuesta": "...",
      "tema": "...",
      "puntaje": 0-5,
      "feedback": "..."
    }},
    ...
  ]
}}

Respuestas del usuario:
{respuestas_usuario}
""")
])

In [19]:
class State(TypedDict):
    user_input: str
    modo: str
    respuestas: List[str]
    pregunta_idx: int
    feedback: Dict[str, Any]
    nivel: str
    fortalezas: List[str]
    debilidades: List[str]
    puntaje_promedio: float
    detalle: List[Dict[str, Any]]
    preguntas_seleccionadas: List[Dict[str, Any]]
    temas: List[str]
    subtemas : Dict[str, Any]
    tema_actual : int

In [9]:
# nodo
def nodo_calificar(state):
    parser = JsonOutputParser()
    chain_quiz = prompt_quiz | llm | parser
    
    respuestas = state.get("respuestas", [])
    preguntas_seleccionadas = state.get("preguntas_seleccionadas", [])
    respuestas_usuario = []
    for idx, pregunta in enumerate(preguntas_seleccionadas):
        if idx < len(respuestas):
            respuestas_usuario.append({
                "pregunta": pregunta["pregunta"],
                "respuesta": respuestas[idx],
                "tema": pregunta["tema"]
            })
    prompt_str = prompt_quiz.format(respuestas_usuario=str(respuestas_usuario))
    data = chain_quiz.invoke(prompt_str)

    resultados = data.get("resultados", [])
    detalle = data.get("detalle", [])
    promedio = sum(resultados) / len(resultados) if resultados else 0
    promedio = round(promedio, 2)
    fortalezas = [d["tema"] for d in detalle if d["puntaje"] >= 4]
    debilidades = [d["tema"] for d in detalle if d["puntaje"] < 4]
    state["feedback"] = data
    state["fortalezas"] = list(set(state.get("fortalezas", []) + fortalezas))
    state["debilidades"] = list(set(state.get("debilidades", []) + debilidades))
    state["puntaje_promedio"] = promedio
    state["detalle"] = detalle
    return state

In [10]:
estado_prueba = State(
    user_input="quiero que me guies paso a paso",
    modo="modo guiado",
    # Lista de respuestas del usuario
    respuestas=["Primero se calcula la probabilidad de no obtener ningún 6. Cada dado tiene 5/6 de probabilidad de no sacar un 6",
                "Una desviación estándar alta indica que los valores del conjunto de datos están muy dispersos respecto a la media. Es decir, hay gran variabilidad entre los datos.",
                "El Teorema de Bayes se usa para actualizar la probabilidad de un evento basándose en nueva evidencia"],

    pregunta_idx=3,  # Indica que ya se respondieron 3 preguntas

    # Las preguntas que fueron seleccionadas y respondidas
    preguntas_seleccionadas=[
        {
            "pregunta": "¿Cuál es la probabilidad de obtener al menos un 6 al lanzar dos dados?",
            "tema": "Probabilidades"
            
        },
        {
            "pregunta": "Qué indica una desviación estándar alta en un conjunto de datos?",
            "tema": "Desviación estándar"
            
        },
        {
            "pregunta": "¿Para qué se utiliza el Teorema de Bayes?",
            "tema": "Teorema de Bayes"
            
        }
    ],

    feedback={},  # Se llenará después de la calificación
    nivel="basico",
    fortalezas=[],  # Se llenará después de la calificación
    debilidades=[],  # Se llenará después de la calificación
    puntaje_promedio=0.0,  # Se calculará después de la calificación
    detalle=[]  # Se llenará después de la calificación
)

# Probar el nodo
estado_actualizado = nodo_calificar(estado_prueba)

# Imprimir los resultados actualizados

#print("\nResultados actualizados:")
#print(f"Puntaje promedio: {estado_actualizado['puntaje_promedio']}")
#print(f"Fortalezas: {estado_actualizado['fortalezas']}")
#print(f"Debilidades: {estado_actualizado['debilidades']}")
#print(f"Detalle: {estado_actualizado['detalle']}")




In [11]:
print(estado_actualizado["feedback"]["detalle"])

[{'pregunta': '¿Cuál es la probabilidad de obtener al menos un 6 al lanzar dos dados?', 'respuesta': 'Primero se calcula la probabilidad de no obtener ningún 6. Cada dado tiene 5/6 de probabilidad de no sacar un 6', 'tema': 'Probabilidades', 'puntaje': 2, 'feedback': 'Buena idea al calcular primero P(no obtener 6)=5/6·5/6, pero la respuesta está incompleta: falta la resta y el resultado final 1−25/36=11/36.'}, {'pregunta': 'Qué indica una desviación estándar alta en un conjunto de datos?', 'respuesta': 'Una desviación estándar alta indica que los valores del conjunto de datos están muy dispersos respecto a la media. Es decir, hay gran variabilidad entre los datos.', 'tema': 'Desviación estándar', 'puntaje': 5, 'feedback': 'Respuesta correcta y precisa: describe adecuadamente que una desviación estándar alta significa mayor dispersión de los datos.'}, {'pregunta': '¿Para qué se utiliza el Teorema de Bayes?', 'respuesta': 'El Teorema de Bayes se usa para actualizar la probabilidad de un 

In [10]:
%pip install langsmith

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install -U langsmith openevals openai

Note: you may need to restart the kernel to use updated packages.


In [17]:
from langsmith import Client

client = Client()

# Programmatically create a dataset in LangSmith
# For other dataset creation methods, see:
# https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_programmatically
# https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_in_application
dataset = client.create_dataset(
    dataset_name="dataset2", description="A sample dataset in LangSmith."
)

# Create examples
examples = [
    {
        "inputs": {"question": "Para qué se utiliza el Teorema de Bayes?"},
        "outputs": {"answer": "El Teorema de Bayes se utiliza para actualizar la probabilidad de un evento basándose en nueva evidencia."},
        
    },
    {
        "inputs": {"question": "Cuál es la diferencia entre una distribución binomial negativa y una distribución de Poisson?"},
        "outputs": {"answer": "La distribución de Poisson modela el número de eventos que ocurren en un intervalo fijo de tiempo o espacio, cuando los eventos ocurren de manera independiente y a una tasa constante. En cambio, la distribución binomial negativa modela el número de ensayos necesarios hasta obtener un número fijo de éxitos, algo que la Poisson no maneja bien."},
    },
]

# Add examples to the dataset
client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['8fb9270d-9d45-49c9-b656-1cdc000fb549',
  '8e2e7320-c48c-4cbf-99e5-90e163bb0c87'],
 'count': 2}

In [18]:
class StateGraph(TypedDict):
    userinput: str
    respuesta: str

In [19]:
def agente_base(state: StateGraph) -> str:
    """
    Función base del agente que devuelve un mensaje de respuesta.
    """
    userinput = state.get("user_input", "")
    respuesta = llm.invoke(userinput)
    state["respuesta"] = respuesta
    return state
    


In [20]:
from langsmith import wrappers
from openai import OpenAI

# Wrap the OpenAI client for LangSmith tracing
openai_client = wrappers.wrap_openai(OpenAI())
      
# Define the application logic you want to evaluate inside a target function
# The SDK will automatically send the inputs from the dataset to your target function
def target(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer the following question accurately"},
            {"role": "user", "content": inputs["question"]},
        ],
    )
    return { "answer": response.choices[0].message.content.strip() }

In [21]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="openai:o3-mini",
        feedback_key="correctness",
    )
    eval_result = evaluator(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs
    )
    return eval_result

In [22]:
# After running the evaluation, a link will be provided to view the results in langsmith
experiment_results = client.evaluate(
    target,
    data="dataset2",
    evaluators=[
        correctness_evaluator,
        # can add multiple evaluators here
    ],
    experiment_prefix="first-eval-in-langsmith",
    max_concurrency=2,
)

e:\Maestría_Eafit2024\Maestria\Semestre_3\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'first-eval-in-langsmith-0c3e5c40' at:
https://smith.langchain.com/o/9b5667fb-0169-4747-b111-d9e2793560e2/datasets/4aabfc06-0244-4152-ae54-2dbbe44b3852/compare?selectedSessions=f16e1d4f-ebcb-4d36-b26a-dd870794f065




2it [00:16,  8.44s/it]


## Evaluacion y desempeño del agente con QAEvalChain

In [1]:
pip install langchain openai


Note: you may need to restart the kernel to use updated packages.


In [21]:
# Simulación de state
state = {
    "respuestas": [
        "Para calcular la probabilidad de un evento dado otro evento",  # ejemplo de respuesta de usuario
    ],
    "preguntas_seleccionadas": [
        {
            "pregunta": "¿Para qué se utiliza el Teorema de Bayes?",
            "respuesta_correcta": "El Teorema de Bayes se utiliza para actualizar la probabilidad de un evento basándose en nueva evidencia."
        }
    ]
}


In [23]:
from langchain.evaluation.qa import QAEvalChain
from langchain.chat_models import ChatOpenAI  # o tu LLM favorito
from langchain.prompts import PromptTemplate

# Tu LLM
llm = ChatOpenAI(model="o4-mini", temperature=1)

# Crear la cadena de evaluación
eval_chain = QAEvalChain.from_llm(
    llm=llm,
    criteria=["correctness"],  # Puedes poner ["correctness", "conciseness", "relevance"] si quieres más dimensiones
    prompt=PromptTemplate.from_template(
        "Dada la pregunta: {query}\n"
        "La respuesta del estudiante: {result}\n"
        "La respuesta correcta esperada: {answer}\n\n"
        "¿La respuesta es correcta? Responde sólo 'Sí' o 'No', y explica en una línea por qué."
    )
)

# Preparar los datos para evaluación
examples = []
for idx, pregunta in enumerate(state["preguntas_seleccionadas"]):
    if idx < len(state["respuestas"]):
        examples.append({
            "query": pregunta["pregunta"],
            "answer": "El Teorema de Bayes se utiliza para actualizar la probabilidad de un evento basándose en nueva evidencia.",  # debes tener la respuesta correcta en tus datos
            "result": state["respuestas"][idx],
        })

# Correr la evaluación
results = []
for example in examples:
    result = eval_chain.evaluate_strings(
        prediction=example["result"],
        input=example["query"],
        reference=example["answer"],
    )
    results.append(result)

# Mostrar resultados
for idx, result in enumerate(results):
    print(f"Pregunta {idx+1}: {examples[idx]['query']}")
    print(f"Respuesta del estudiante: {examples[idx]['result']}")
    print(f"Respuesta esperada: {examples[idx]['answer']}")
    print(f"Evaluación: {result['reasoning']}\n")


Pregunta 1: ¿Para qué se utiliza el Teorema de Bayes?
Respuesta del estudiante: Para calcular la probabilidad de un evento dado otro evento
Respuesta esperada: El Teorema de Bayes se utiliza para actualizar la probabilidad de un evento basándose en nueva evidencia.
Evaluación: No: describe la probabilidad condicional, pero no menciona la actualización de la probabilidad con nueva evidencia.



## Evaluandolo con dataset csv

In [28]:
import pandas as pd
from langchain.evaluation.qa import QAEvalChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

# Cargar el CSV
df = pd.read_csv(r"E:\Maestría_Eafit2024\Maestria\Semestre_3\NLP\custom_agent-main\custom_agent-main\Evaluacion\eval_dataset.csv") 

# Configurar el LLM
llm = ChatOpenAI(model="o4-mini", temperature=1)

# Crear la cadena de evaluación
eval_chain = QAEvalChain.from_llm(
    llm=llm,
    criteria=["correctness"],
    prompt=PromptTemplate.from_template(
        "Dada la pregunta: {query}\n"
        "La respuesta del estudiante: {result}\n"
        "La respuesta correcta esperada: {answer}\n\n"
        "¿La respuesta es correcta? Responde sólo 'Sí' o 'No', y explica en una línea por qué."
    )
)

# Preparar ejemplos
examples = df.to_dict(orient="records")

# Evaluar
results = []
for example in examples:
    result = eval_chain.evaluate_strings(
        prediction=example["result"],
        input=example["query"],
        reference=example["answer"],
    )
    results.append(result)

# Agregar los resultados al dataframe
df["evaluacion"] = [r["reasoning"] for r in results]

# Mostrar resultados
print(df)

# Opcional: guardar resultados
df.to_csv("resultados_evaluacion.csv", index=False)


                                       query  \
0  ¿Para qué se utiliza el Teorema de Bayes?   
1           ¿Qué es una distribución normal?   

                                              answer  \
0  El Teorema de Bayes se utiliza para actualizar...   
1  Una distribución de probabilidad continua que ...   

                                              result  \
0  Para calcular la probabilidad de un evento dad...   
1                      Una curva en forma de campana   

                                          evaluacion  
0  No, porque solo menciona calcular probabilidad...  
1  No, porque solo menciona la forma de campana y...  


In [6]:
from dotenv import load_dotenv
load_dotenv()


True

In [7]:
from langchain.chat_models import ChatOpenAI

llm_respuestas = ChatOpenAI(
    model="gpt-3.5-turbo",  # o el modelo que quieras de OpenAI
    temperature=0.7
)


In [ ]:
import pandas as pd
from langchain.evaluation.qa import QAEvalChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Cargar el CSV
df = pd.read_csv(r"E:\Maestría_Eafit2024\Maestria\Semestre_3\NLP\custom_agent-main\custom_agent-main\Evaluacion\eval_dataset.csv") 

# Configurar los LLMs
llm_respuestas = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5)  # Este genera la respuesta del estudiante
llm_evaluador = ChatOpenAI(model="o4-mini", temperature=1)           # Este evalúa la respuesta

# Cadena para generar respuestas automáticas
prompt_responder = PromptTemplate.from_template(
    "Responde de manera breve y adecuada la siguiente pregunta de estadística:\n\n{query}"
)
chain_responder = LLMChain(llm=llm_respuestas, prompt=prompt_responder)

# Cadena de evaluación
eval_chain = QAEvalChain.from_llm(
    llm=llm_evaluador,
    criteria=["correctness"],
    prompt=PromptTemplate.from_template(
        "Dada la pregunta: {query}\n"
        "La respuesta del estudiante: {result}\n"
        "La respuesta correcta esperada: {answer}\n\n"
        "¿La respuesta es correcta? Responde sólo 'Sí' o 'No', y explica en una línea por qué."
    )
)

# Preparar ejemplos
examples = df.to_dict(orient="records")

# Generar respuestas y evaluar
respuestas_generadas = []
evaluaciones = []

for example in examples:
    # 1. Generar respuesta automática
    generated = chain_responder.invoke({"query": example["query"]})
    respuesta_estudiante = generated["text"]
    
    # 2. Evaluar la respuesta
    result = eval_chain.evaluate_strings(
        prediction=respuesta_estudiante,
        input=example["query"],
        reference=example["answer"],
    )
    
    # Guardar resultados
    respuestas_generadas.append(respuesta_estudiante)
    evaluaciones.append(result["reasoning"])

# Agregar resultados al dataframe
df["respuesta_estudiante"] = respuestas_generadas
df["evaluacion"] = evaluaciones

# Mostrar resultados
print(df)

# Guardar resultados
df.to_csv(r"E:\Maestría_Eafit2024\Maestria\Semestre_3\NLP\custom_agent-main\custom_agent-main\Evaluacion\resultados_evaluacion.csv", index=False)


                                       query  \
0  ¿Para qué se utiliza el Teorema de Bayes?   
1           ¿Qué es una distribución normal?   

                                              answer  \
0  El Teorema de Bayes se utiliza para actualizar...   
1  Una distribución de probabilidad continua que ...   

                                respuesta_estudiante  \
0  El Teorema de Bayes se utiliza para actualizar...   
1  Una distribución normal es un tipo de distribu...   

                                          evaluacion  
0  Sí. Porque describe correctamente la actualiza...  
1  Sí. Porque describe correctamente su forma de ...  


In [12]:
import pandas as pd
from dotenv import load_dotenv
from langchain.evaluation.qa import QAEvalChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Cargar .env
load_dotenv()

# Cargar dataset
df = pd.read_csv(r"E:\Maestría_Eafit2024\Maestria\Semestre_3\NLP\custom_agent-main\custom_agent-main\Evaluacion\eval_dataset.csv")

# LLM para evaluación
llm_evaluacion = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# LLM para simular respuestas de estudiantes
llm_respuestas = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

# Prompt de estudiante
student_prompt = PromptTemplate.from_template(
    """Imagina que eres un estudiante que responde una evaluación de estadística.
A veces respondes bien y otras veces te equivocas o das respuestas incompletas.

Pregunta: {pregunta}

Responde en una o dos frases, como un estudiante humano. No siempre respondas perfectamente."""
)

chain_respuestas = LLMChain(llm=llm_respuestas, prompt=student_prompt)

# Crear la cadena de evaluación
eval_chain = QAEvalChain.from_llm(
    llm=llm_evaluacion,
    criteria=["correctness"],
    prompt=PromptTemplate.from_template(
        "Dada la pregunta: {query}\n"
        "La respuesta del estudiante: {result}\n"
        "La respuesta correcta esperada: {answer}\n\n"
        "¿La respuesta es correcta? Responde sólo 'Sí' o 'No', y explica en una línea por qué."
    )
)

# Preparar ejemplos
examples = df.to_dict(orient="records")

# Evaluar
results = []
respuestas_generadas = []

for example in examples:
    # Generar respuesta de estudiante
    respuesta = chain_respuestas.invoke({"pregunta": example["query"]})
    respuesta_estudiante = respuesta["text"]

    respuestas_generadas.append(respuesta_estudiante)

    # Evaluar
    result = eval_chain.evaluate_strings(
        prediction=respuesta_estudiante,
        input=example["query"],
        reference=example["answer"],
    )
    results.append(result)

# Agregar las respuestas generadas y evaluaciones al dataframe
df["respuesta_estudiante"] = respuestas_generadas
df["evaluacion"] = [r["reasoning"] for r in results]

# Mostrar resultados
print(df)

# Guardar
df.to_csv("resultados_evaluacion_simulada.csv", index=False)




                                       query  \
0  ¿Para qué se utiliza el Teorema de Bayes?   
1           ¿Qué es una distribución normal?   

                                              answer  \
0  El Teorema de Bayes se utiliza para actualizar...   
1  Una distribución de probabilidad continua que ...   

                                              result  \
0  Para calcular la probabilidad de un evento dad...   
1                      Una curva en forma de campana   

                                respuesta_estudiante  \
0  El Teorema de Bayes se utiliza para calcular l...   
1  Una distribución normal es una distribución de...   

                                          evaluacion  
0  Sí. La respuesta del estudiante explica de man...  
1  No. La respuesta del estudiante describe la fo...  
